### Setup

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="3"

import torch, transformers, datasets, diffusers, peft
import random
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import os
import json
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance

import transformers, datasets, diffusers
from huggingface_hub import login

from diffusers import DDPMScheduler, StableDiffusionPipeline, StableDiffusionPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   
print('package version : ',torch.__version__, transformers.__version__, datasets.__version__, diffusers.__version__, peft.__version__)

# CONFIG
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_SEED = 42

# TRAIN PARAMETER
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 1e-04
TRAIN_DATA_SIZE = 1800
TEST_DATA_SIZE = 200

# PATH
CONFIG_PATH = '../config.json'
## DATA
# TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_KFashion' # change path
# TEST_LABEL_FOLDER = '../Data/Total/Test_Label_CLIP_Summarize' # change path
# TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_Nike' # change path
# TEST_LABEL_FOLDER = '../Data/Total/Test_Label_Nike' # change path
TEST_IMAGE_FOLDER = '../Data/Total/Test_Image_Zara' # change path
TEST_LABEL_FOLDER = '../Data/Total/Test_Label_Zara' # change path
TEST_IMAGE_FILE = sorted([f for f in os.listdir(TEST_IMAGE_FOLDER) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
TEST_LABEL_FILE = sorted([f for f in os.listdir(TEST_LABEL_FOLDER) if f.endswith('.json')])

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"

# HUGGINGFACE
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
HUGGING_FACE_TOKEN = config.get("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

2025-03-03 22:49:44.158594: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-03 22:49:44.185248: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-03 22:49:44.670054: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


device :  cuda
package version :  2.4.1+cu124 4.46.2 3.0.1 0.33.0.dev0 0.7.0


### Load Model

In [2]:
# 저장된 모델 불러오기
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250228_165330'
pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True)
pipe.to("cuda")

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용
vae = pipe.vae.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용
unet = pipe.unet.to(torch.float16).to(device)  # 🔥 float16 + CUDA 적용

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/gayeon38/anaconda3/envs/vlm-env/lib/python3.8/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


### Preprocess Data

In [3]:
# 주어진 데이터 샘플에서 text 컬럼을 토크나이징하여 토큰 ID tensor를 반환하는 함수.
def tokenize_captions(examples, caption_column='text', is_train=True):
    captions = []
    for caption in examples[caption_column]:
        # 캡션이 하나인 경우
        if isinstance(caption, str):
            captions.append(caption)
        # 캡션이 하나 이상인 경우 아무거나 하나 선택
        elif isinstance(caption, (list, np.ndarray)):
            # take a random caption if there are multiple
            captions.append(random.choice(caption) if is_train else caption[0])
        else:
            raise ValueError(
                f"Caption column `{caption_column}` should contain either strings or lists of strings."
            )
    inputs = tokenizer(
        captions, max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt"
    )

    return inputs.input_ids

# 이미지 데이터를 학습용으로 전처리하는 변환 파이프라인
transforms = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(IMG_SIZE) if True else transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip() if True else transforms.Lambda(lambda x: x),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]), # (0,1)->(-1,1)
    ]
)

# 이미지 데이터를 RGB로 변환 후 전처리하고, text 컬럼을 토크나이징하여 학습 데이터셋을 구성하는 함수.
def preprocess_data(examples, image_column='image'):
    images = [image.convert("RGB") for image in examples[image_column]]
    # 이미지 전처리
    examples["pixel_values"] = [transforms(image) for image in images]
    # 텍스트 전처리
    examples["input_ids"] = tokenize_captions(examples)
    return examples

# 배치 단위로 이미지와 토큰 ID를 스택하여 PyTorch 텐서 형태로 변환하는 함수.
def collate_fn(examples):
    # (C, H, W), ..., (C, H, W) -> stack -> (N, C, H, W): N으로 스택
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    # tensor.contiguous()와 동일
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()
    # {(77,), ..., (77,)}_N개 -> stack -> (N, 77)
    input_ids = torch.stack([example["input_ids"] for example in examples])

    return {"pixel_values": pixel_values, "input_ids": input_ids}


In [4]:
data = []
for image_file, label_file in zip(TEST_IMAGE_FILE, TEST_LABEL_FILE):
    image_path = os.path.join(TEST_IMAGE_FOLDER, image_file)
    label_path = os.path.join(TEST_LABEL_FOLDER, label_file)
    # 이미지 파일 열기
    with open(image_path, 'rb') as image:
        image_data = Image.open(image)
        image_data = image_data.convert('RGB')
    # 라벨 파일 열기
    with open(label_path, 'r') as label:
        label_data = json.load(label)
    # 이미지와 라벨 데이터 묶기
    data.append({
        'image':image_data,
        'text':label_data
    })
    
# Dataset으로 변환
if TEST_IMAGE_FOLDER[25:] == "KFashion":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['summary'] for item in data]
    })
elif TEST_IMAGE_FOLDER[25:] == "Nike" or TEST_IMAGE_FOLDER[25:] == "Zara":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [
            " | ".join([
                str(item['text']['Prompt']), 
                str(item['text']['Input']), 
                str(item['text']['Add_Info'])
            ])
            for item in data
        ]
    })

# Test DatasetDict 생성 및 전처리 적용
test_dataset = DatasetDict({'test': dataset})
test_dataset = test_dataset.with_transform(preprocess_data)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset['test'],
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=BATCH_SIZE
)

print(test_dataset)

DatasetDict({
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 200
    })
})


### Evaluate

In [5]:
# 텍스트 조건으로부터 이미지를 생성하는 간단한 샘플링 함수
def generate_images_from_batch(batch, num_inference_steps=50):
    generated_images = []

    # 입력 텐서를 device로 이동 및 텍스트 인코딩
    input_ids = batch["input_ids"].to(device)
    encoder_hidden_states = text_encoder(input_ids)[0].to(torch.float16)  # 🔥 float16 변환
    batch_size = input_ids.shape[0]

    # 배치의 이미지 shape에서 height와 width 추출
    _, _, H, W = batch["pixel_values"].shape
    latent_shape = (1, vae.config.latent_channels, H // 8, W // 8)

    for i in range(batch_size):
        # 개별 예제에 대한 텍스트 임베딩 추출
        single_embedding = encoder_hidden_states[i].unsqueeze(0).to(torch.float16)  # 🔥 float16 변환
        # 정규분포로부터 latent 초기화
        latent = torch.randn(latent_shape, device=device, dtype=torch.float16)  # 🔥 float16 변환

        # Reverse diffusion sampling
        for t in reversed(range(num_inference_steps)):
            timestep = torch.tensor([t], device=device).long()  # ✅ long 유지 (float16 변환 X)
            
            with torch.no_grad():
                latent_model_input = latent.to(torch.float16)  # 🔥 float16 변환
                noise_pred = unet(latent_model_input, timestep, single_embedding).sample
                latent = noise_scheduler.step(noise_pred, timestep, latent).prev_sample

        # latent를 이미지로 디코딩
        with torch.no_grad():
            image = vae.decode((latent / vae.config.scaling_factor).to(torch.float16)).sample  # 🔥 float16 변환
            image = torch.clamp(image, 0, 1)
        generated_images.append(image)

    # 배치 차원으로 이미지들을 결합
    generated_images = torch.cat(generated_images, dim=0)
    return generated_images

In [6]:
# FID metric 초기화
fid_metric = FrechetInceptionDistance(feature=64).to(device)
fid_metric.reset()
unet.eval()
text_encoder.eval()
vae.eval()

for step, batch in enumerate(tqdm(test_dataloader, desc="Evaluating on test set")):
    with torch.no_grad():
        # 텍스트 조건에 따른 이미지 생성
        generated_imgs = generate_images_from_batch(batch, num_inference_steps=50)
        # 생성된 이미지는 [0,1] 범위의 float이므로, [0,255] 범위의 uint8로 변환
        generated_imgs_uint8 = (generated_imgs * 255).round().to(torch.uint8)
        
        # 실제 이미지: 전처리 시 Normalize([0.5],[0.5])가 적용되어 [-1,1] 범위이므로, [0,1]로 복원 후 변환
        real_imgs = batch["pixel_values"].to(device)
        real_imgs_denorm = real_imgs * 0.5 + 0.5
        real_imgs_uint8 = (real_imgs_denorm * 255).round().to(torch.uint8)
        
        # FID metric 업데이트
        fid_metric.update(generated_imgs_uint8, real=False)
        fid_metric.update(real_imgs_uint8, real=True)

fid_score = fid_metric.compute()
print(f'Final test FID score: {fid_score:.5f}')

Evaluating on test set: 100%|██████████| 50/50 [05:28<00:00,  6.57s/it]

Final test FID score: 400.95721
